# Modelos: de `Sequential` a `nn.Module`

En el módulo 1 casi todos los modelos se armaron con `nn.Sequential`, que aplica sus capas una tras otra. Eso alcanza mientras los datos pasen en fila por las capas. Para cualquier otra cosa, como usar una salida intermedia, combinar dos caminos o revisar formas a mitad del cálculo, el modelo se escribe como una clase que hereda de `nn.Module`. El proyecto de punta a punta del módulo 1 ya usó una. Aquí vemos qué hace cada parte y qué se rompe cuando falta algo.

Usamos el modelo de MNIST que se entrena en `04-clasificacion-de-imagenes`: aplanar la imagen, una capa oculta de 128 neuronas con ReLU y una salida de 10 números, uno por dígito.

**Contenido**

1. Preparación
2. El mismo modelo, de dos formas
3. `__init__` declara, `forward` conecta
4. Por qué `super().__init__()` no es opcional
5. `model(x)` y no `model.forward(x)`
6. De imagen a vector: `Flatten`
7. Contar parámetros
8. Lo que `Sequential` no puede hacer
9. `model.eval()` y `torch.no_grad()`
10. Resumen y ejercicios

## 1. Preparación

Tomamos un lote de MNIST armado como en `00-datos`. Aquí solo sirve para ver formas: no entrenamos nada.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)

transformacion = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
entrenamiento = datasets.MNIST(root="../data", train=True, download=True, transform=transformacion)
x, y = next(iter(DataLoader(entrenamiento, batch_size=64, shuffle=True)))

print("imágenes:", tuple(x.shape), "  etiquetas:", tuple(y.shape))

imágenes: (64, 1, 28, 28)   etiquetas: (64,)


## 2. El mismo modelo, de dos formas

Con `Sequential`:

In [2]:
secuencial = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
print(secuencial)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=10, bias=True)
)


Como clase:

In [3]:
class Red(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplanar = nn.Flatten()
        self.oculta = nn.Linear(784, 128)
        self.activacion = nn.ReLU()
        self.salida = nn.Linear(128, 10)

    def forward(self, x):
        x = self.aplanar(x)          # [lote, 1, 28, 28] -> [lote, 784]
        x = self.oculta(x)           # [lote, 784] -> [lote, 128]
        x = self.activacion(x)
        return self.salida(x)        # [lote, 128] -> [lote, 10]


red = Red()
print(red)

Red(
  (aplanar): Flatten(start_dim=1, end_dim=-1)
  (oculta): Linear(in_features=784, out_features=128, bias=True)
  (activacion): ReLU()
  (salida): Linear(in_features=128, out_features=10, bias=True)
)


Los dos tienen las mismas capas, con los mismos tamaños. Si se copian los pesos de uno al otro, calculan exactamente lo mismo:

In [4]:
with torch.no_grad():
    red.oculta.weight.copy_(secuencial[1].weight)
    red.oculta.bias.copy_(secuencial[1].bias)
    red.salida.weight.copy_(secuencial[3].weight)
    red.salida.bias.copy_(secuencial[3].bias)

print("forma de la salida:", tuple(red(x).shape))
print("¿Misma salida?", torch.equal(red(x), secuencial(x)))

forma de la salida: (64, 10)
¿Misma salida? True


La clase es más larga, pero cada capa tiene nombre (`red.oculta` en vez de `secuencial[1]`) y el orden de los cálculos queda escrito en `forward`, donde se puede cambiar.

## 3. `__init__` declara, `forward` conecta

- **`__init__`** crea las capas y las guarda como atributos. Es como reunir las herramientas antes de empezar: todavía no pasa ningún dato.
- **`forward`** recibe la entrada y dice en qué orden se usan las capas. Se ejecuta cada vez que el modelo procesa un lote.

Lo que se guarda en `__init__` queda registrado en el modelo. `named_parameters()` recorre todos los parámetros entrenables, cada uno con el nombre del atributo donde vive:

In [5]:
for nombre, parametro in red.named_parameters():
    print(f"{nombre:15s} {str(tuple(parametro.shape)):12s} requires_grad={parametro.requires_grad}")

oculta.weight   (128, 784)   requires_grad=True
oculta.bias     (128,)       requires_grad=True
salida.weight   (10, 128)    requires_grad=True
salida.bias     (10,)        requires_grad=True


Esta es la lista que recibe el optimizador cuando se le pasa `red.parameters()`, y la que se guarda en disco con `state_dict()`. `Flatten` y `ReLU` no aparecen porque no tienen parámetros: son operaciones fijas.

## 4. Por qué `super().__init__()` no es opcional

`super().__init__()` ejecuta el `__init__` de `nn.Module`, que crea los registros donde el módulo anota sus capas y parámetros. Sin esa línea los registros no existen, y el primer intento de guardar una capa falla:

In [6]:
class SinSuper(nn.Module):
    def __init__(self):
        self.oculta = nn.Linear(784, 128)


try:
    SinSuper()
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: cannot assign module before Module.__init__() call


El mensaje dice qué falta: no se puede asignar un módulo antes de llamar a `Module.__init__()`.

Detrás hay un mecanismo de `nn.Module`. Cuando se asigna un atributo que es una capa, como `self.oculta = nn.Linear(...)`, el módulo no lo guarda como un atributo común sino en su registro de submódulos. Los atributos que empiezan con `_` son internos y no se usan en código normal, pero sirven para mirar ese registro:

In [7]:
print("submódulos de red:", list(red._modules.keys()))
print("parámetros propios de red.oculta:", list(red.oculta._parameters.keys()))

submódulos de red: ['aplanar', 'oculta', 'activacion', 'salida']
parámetros propios de red.oculta: ['weight', 'bias']


El registro solo reconoce capas asignadas directamente como atributos. Una trampa común es guardar las capas en una lista de Python: no da ningún error, pero el modelo queda sin parámetros registrados.

In [8]:
class ConLista(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = [nn.Linear(784, 128), nn.Linear(128, 10)]      # lista común de Python

    def forward(self, x):
        x = torch.relu(self.capas[0](x.flatten(1)))
        return self.capas[1](x)


class ConModuleList(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.ModuleList([nn.Linear(784, 128), nn.Linear(128, 10)])

    def forward(self, x):
        x = torch.relu(self.capas[0](x.flatten(1)))
        return self.capas[1](x)


for clase in [ConLista, ConModuleList]:
    modelo = clase()
    print(f"{clase.__name__:14s} salida {tuple(modelo(x).shape)}   parámetros registrados: {len(list(modelo.parameters()))}")

ConLista       salida (64, 10)   parámetros registrados: 0
ConModuleList  salida (64, 10)   parámetros registrados: 4


Los dos calculan una salida de la forma correcta, pero el primero no tiene ningún parámetro registrado. Aquí el optimizador lo detecta, porque recibe una lista vacía:

In [9]:
try:
    torch.optim.Adam(ConLista().parameters(), lr=1e-3)
except ValueError as error:
    print("ValueError:", error)

ValueError: optimizer got an empty parameter list


Si el modelo tuviera además alguna capa bien registrada, ni siquiera habría error: el optimizador entrenaría esa capa y las de la lista se quedarían con sus pesos aleatorios para siempre. `model.to(device)` y `state_dict()` tampoco verían las capas de la lista. `nn.ModuleList` y `nn.ModuleDict` son la lista y el diccionario que sí registran lo que contienen.

## 5. `model(x)` y no `model.forward(x)`

`forward` se define pero no se llama directamente. Se escribe `red(x)`, y `nn.Module` llama a `forward` por dentro. Lo que agrega en el camino son los **hooks**: funciones que se enganchan a un módulo para que se ejecuten antes o después de su `forward`. Las usan las herramientas que inspeccionan modelos, como las que imprimen un resumen de capas o las que miden tiempos.

Enganchamos un hook al modelo y lo llamamos de las dos maneras:

In [10]:
def avisar(modulo, entrada, salida):
    print(f"   hook: {modulo.__class__.__name__} entregó una salida de forma {tuple(salida.shape)}")


gancho = red.register_forward_hook(avisar)

print("red(x):")
a = red(x)
print("red.forward(x):")
b = red.forward(x)

gancho.remove()

print("\n¿Mismo resultado?", torch.equal(a, b))
print("¿Las dos salidas permiten calcular gradientes?", a.grad_fn is not None, b.grad_fn is not None)

red(x):
   hook: Red entregó una salida de forma (64, 10)
red.forward(x):

¿Mismo resultado? True
¿Las dos salidas permiten calcular gradientes? True True


Llamando a `forward` directamente el resultado es el mismo, y los gradientes se pueden calcular igual: los dos tensores tienen `grad_fn`, porque el registro para autograd lo hace cada operación, no `nn.Module`. Lo que se pierde es el hook. Este modelo no tenía ninguno y no pasaría nada, pero el día que alguien, o una librería, le enganche uno, `model.forward(x)` se lo va a saltar sin avisar. Por eso la regla es siempre `model(x)`.

## 6. De imagen a vector: `Flatten`

Una capa `nn.Linear` trabaja sobre la **última** dimensión del tensor: `nn.Linear(784, 128)` espera que la última dimensión mida 784. Un lote de MNIST tiene forma `[64, 1, 28, 28]` y su última dimensión mide 28. `nn.Flatten()` junta todas las dimensiones salvo la primera, que es la del lote:

In [11]:
aplanar = nn.Flatten()
print("antes  :", tuple(x.shape))
print("después:", tuple(aplanar(x).shape), "  porque 1 × 28 × 28 =", 1 * 28 * 28)

antes  : (64, 1, 28, 28)
después: (64, 784)   porque 1 × 28 × 28 = 784


`nn.Flatten()` empieza a aplanar en la dimensión 1 y deja intacta la 0. La función `torch.flatten(x)`, en cambio, empieza en la 0 si no se le dice otra cosa, y aplasta el lote entero en un solo vector. Las variantes:

In [12]:
print("nn.Flatten()(x)        :", tuple(nn.Flatten()(x).shape))
print("torch.flatten(x)       :", tuple(torch.flatten(x).shape))
print("torch.flatten(x, 1)    :", tuple(torch.flatten(x, 1).shape))
print("x.view(x.shape[0], -1) :", tuple(x.view(x.shape[0], -1).shape))

nn.Flatten()(x)        : (64, 784)
torch.flatten(x)       : (50176,)
torch.flatten(x, 1)    : (64, 784)
x.view(x.shape[0], -1) : (64, 784)


Sin `Flatten`, la capa lineal recibe la imagen con su forma original. El error trae números que vale la pena saber leer:

In [13]:
sin_aplanar = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

try:
    sin_aplanar(x)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1792x28 and 784x128)


`mat1` es la entrada y `mat2` son los pesos de la capa, traspuestos. La capa trató todas las dimensiones menos la última como filas: 64 × 1 × 28 = 1792 filas de 28 valores. Para multiplicarlas por una matriz de 784 × 128, cada fila tendría que traer 784 valores, y trae 28. Cuando un error de formas muestra un número raro como 1792, casi siempre es el producto de varias dimensiones que no se aplanaron.

Para seguir las formas capa por capa, los hooks vuelven a servir. Esta vez enganchamos uno en cada capa interna:

In [14]:
def anotar_forma(modulo, entrada, salida):
    print(f"{modulo.__class__.__name__:8s} {str(tuple(entrada[0].shape)):16s} -> {tuple(salida.shape)}")


ganchos = [capa.register_forward_hook(anotar_forma) for capa in red.children()]
_ = red(x)
for gancho in ganchos:
    gancho.remove()

Flatten  (64, 1, 28, 28)  -> (64, 784)
Linear   (64, 784)        -> (64, 128)
ReLU     (64, 128)        -> (64, 128)
Linear   (64, 128)        -> (64, 10)


## 7. Contar parámetros

Cada `nn.Linear(entradas, salidas)` tiene una matriz de pesos de `salidas × entradas` y un sesgo por salida. Para la red 784 → 128 → 10:

| Capa | Pesos | Sesgos | Total |
|---|---|---|---|
| `oculta` | 784 × 128 = 100.352 | 128 | 100.480 |
| `salida` | 128 × 10 = 1.280 | 10 | 1.290 |
| **Total** | | | **101.770** |

In [15]:
total = sum(p.numel() for p in red.parameters())
print("parámetros:", total)
for nombre, modulo in red.named_children():
    print(f"   {nombre:10s} {sum(p.numel() for p in modulo.parameters()):7d}")
print(f"\nmemoria de los pesos en float32: {total * 4 / 1e3:.0f} KB")

parámetros: 101770
   aplanar          0
   oculta      100480
   activacion       0
   salida        1290

memoria de los pesos en float32: 407 KB


El 99% de los parámetros está en la primera capa, porque conecta cada uno de los 784 píxeles con cada una de las 128 neuronas. En una red densa sobre imágenes eso es lo normal, y es una de las razones por las que más adelante se usan capas convolucionales, que reutilizan los mismos pesos en toda la imagen.

## 8. Lo que `Sequential` no puede hacer

`forward` es código de Python común, así que puede hacer cualquier cosa: usar un `if`, llamar dos veces a la misma capa o devolver varios resultados. Dos ejemplos.

**Devolver una representación intermedia.** A veces interesa lo que calcula la capa oculta y no solo la salida, para visualizarlo o para reutilizarlo en otro problema. Heredando de `Red` se reutiliza su `__init__` y solo se reescribe `forward`:

In [16]:
class RedConOculta(Red):
    def forward(self, x, devolver_oculta=False):
        oculta = self.activacion(self.oculta(self.aplanar(x)))
        salida = self.salida(oculta)
        if devolver_oculta:
            return salida, oculta
        return salida


modelo = RedConOculta()
salida, oculta = modelo(x, devolver_oculta=True)
print("salida:", tuple(salida.shape), "  capa oculta:", tuple(oculta.shape))
print(f"valores de la capa oculta que la ReLU dejó en cero: {(oculta == 0).float().mean():.0%}")

salida: (64, 10)   capa oculta: (64, 128)
valores de la capa oculta que la ReLU dejó en cero: 51%


**Una conexión residual.** Sumarle a la salida de una capa su propia entrada: `x + capa(x)`. Es la idea de las redes residuales (ResNet), que permitió entrenar redes de cientos de capas. Con `Sequential` no se puede escribir, porque cada capa solo ve la salida de la anterior:

In [17]:
class BloqueResidual(nn.Module):
    def __init__(self, ancho):
        super().__init__()
        self.capa = nn.Linear(ancho, ancho)

    def forward(self, x):
        return x + torch.relu(self.capa(x))     # la entrada se suma a la salida


class RedResidual(nn.Module):
    def __init__(self):
        super().__init__()
        self.entrada = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU())
        self.bloques = nn.Sequential(BloqueResidual(128), BloqueResidual(128))
        self.salida = nn.Linear(128, 10)

    def forward(self, x):
        return self.salida(self.bloques(self.entrada(x)))


residual = RedResidual()
print(residual)
print("\nforma de la salida:", tuple(residual(x).shape))

RedResidual(
  (entrada): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
  )
  (bloques): Sequential(
    (0): BloqueResidual(
      (capa): Linear(in_features=128, out_features=128, bias=True)
    )
    (1): BloqueResidual(
      (capa): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (salida): Linear(in_features=128, out_features=10, bias=True)
)

forma de la salida: (64, 10)


El ejemplo también muestra que las piezas se combinan: módulos propios dentro de un `Sequential`, dentro de otro módulo propio. Todo queda registrado, porque cada pieza es un `nn.Module` asignado como atributo.

## 9. `model.eval()` y `torch.no_grad()`

Para evaluar se usan dos cosas, y sus nombres confunden un poco.

**`model.eval()` no evalúa nada.** Solo cambia una bandera, `training`, en el modelo y en todas sus capas. Algunas capas se comportan distinto al entrenar y al predecir. La más común es `nn.Dropout(p)`: al entrenar apaga al azar una fracción `p` de los valores que recibe (y multiplica el resto por $1/(1-p)$ para compensar), y al predecir no hace nada.

In [18]:
torch.manual_seed(0)
con_dropout = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 10))
imagen = x[:1]

con_dropout.train()
print("modo entrenamiento: training =", con_dropout.training, "  en el dropout:", con_dropout[3].training)
print("   ¿dos llamadas dan lo mismo?", torch.equal(con_dropout(imagen), con_dropout(imagen)))

con_dropout.eval()
print("modo evaluación   : training =", con_dropout.training, " en el dropout:", con_dropout[3].training)
print("   ¿dos llamadas dan lo mismo?", torch.equal(con_dropout(imagen), con_dropout(imagen)))

modo entrenamiento: training =

 True   en el dropout: True
   ¿dos llamadas dan lo mismo? False
modo evaluación   : training = False  en el dropout: False
   ¿dos llamadas dan lo mismo? True


En modo entrenamiento, la misma imagen da dos salidas distintas porque el dropout sortea valores distintos en cada llamada. En modo evaluación el modelo es determinista, y `eval()` llegó hasta la capa de dropout aunque se llamó sobre el modelo completo.

Si se olvida `model.eval()` antes de evaluar, se evalúa con valores apagados al azar y la exactitud sale peor y distinta cada vez. Si se olvida volver a `model.train()` antes de seguir entrenando, el dropout queda desactivado. La red de este módulo no tiene dropout, así que no se notaría, pero la costumbre de llamar a los dos evita el error cuando el modelo cambie.

**`torch.no_grad()` no cambia el modo.** Hace otra cosa: apaga el registro de autograd dentro del bloque `with`. Para poder calcular gradientes, PyTorch retiene los resultados intermedios de cada operación hasta que se llama a `backward()`. Al evaluar no habrá `backward()`, así que retenerlos es memoria perdida.

Para medir cuánta, `torch.autograd.graph.saved_tensors_hooks` permite ver cada tensor que autograd retiene. Sumamos sus tamaños con un lote de 1.000 imágenes, sin contar los pesos, que están en memoria de todas formas:

In [19]:
def memoria_retenida(modelo, entrada):
    pesos = {p.data_ptr() for p in modelo.parameters()}
    retenidos = {}

    def anotar(tensor):
        if tensor.data_ptr() not in pesos:
            retenidos[tensor.data_ptr()] = tensor.numel() * tensor.element_size()
        return tensor

    with torch.autograd.graph.saved_tensors_hooks(anotar, lambda tensor: tensor):
        salida = modelo(entrada)
    return salida, sum(retenidos.values())


lote_grande = torch.randn(1000, 1, 28, 28)

salida, retenido = memoria_retenida(red, lote_grande)
print(f"normal       : requires_grad={salida.requires_grad}   retenido para backward: {retenido / 1e6:.2f} MB")

with torch.no_grad():
    salida, retenido = memoria_retenida(red, lote_grande)
print(f"con no_grad(): requires_grad={salida.requires_grad}  retenido para backward: {retenido / 1e6:.2f} MB")

normal       : requires_grad=True   retenido para backward: 3.65 MB


con no_grad(): requires_grad=False  retenido para backward: 0.00 MB


Sin `no_grad()`, autograd retiene la entrada aplanada (1.000 × 784 valores, 3.1 MB) y la salida de la capa oculta (1.000 × 128, 0.5 MB), que necesitaría para calcular los gradientes. Con `no_grad()` no retiene nada, y la salida ni siquiera tiene `requires_grad`. En esta red son pocos megabytes, pero crece con el tamaño del lote y con cada capa: en una red convolucional profunda, lo retenido para `backward()` suele ocupar mucho más que los pesos. Por eso el cargador de prueba de `00-datos` usa lotes de 1.000 y el de entrenamiento lotes de 64.

`eval()` y `no_grad()` son independientes: `eval()` no apaga los gradientes y `no_grad()` no cambia el modo. Para evaluar se usan los dos.

| | con `torch.no_grad()` | sin `torch.no_grad()` |
|---|---|---|
| **`model.eval()`** | evaluar: lo correcto | resultados correctos, pero retiene memoria que no se usa |
| **`model.train()`** | resultados alterados por capas como dropout | entrenar: lo correcto |

`torch.inference_mode()` es una versión más estricta de `no_grad()`: además de no registrar operaciones, marca los tensores creados para que nunca puedan entrar a un cálculo con gradientes, y a cambio es un poco más rápida. Para evaluar se puede usar cualquiera de las dos.

## 10. Resumen

- Un modelo como clase hereda de **`nn.Module`**: en **`__init__`** se crean las capas y en **`forward`** se dice cómo pasan los datos por ellas.
- **`super().__init__()`** crea los registros de capas y parámetros. Sin esa línea, asignar una capa da `AttributeError`.
- Solo se registran las capas asignadas como atributos, o guardadas en **`nn.ModuleList`** o **`nn.ModuleDict`**. Las que están en una lista común no reciben entrenamiento, y a veces no hay ningún error que lo avise.
- Se llama **`model(x)`**, no `model.forward(x)`. Los gradientes funcionan igual, pero `forward` directo se salta los hooks.
- `nn.Linear` trabaja sobre la última dimensión. **`nn.Flatten()`** convierte `[lote, 1, 28, 28]` en `[lote, 784]` sin tocar la dimensión del lote. Un número raro en un error de formas, como 1792, suele ser el producto de dimensiones sin aplanar.
- `nn.Linear(a, b)` tiene `a × b + b` parámetros. La red 784 → 128 → 10 tiene **101.770**, casi todos en la primera capa.
- **`forward`** es código de Python: puede devolver resultados intermedios o sumar la entrada a la salida, cosas que `Sequential` no permite.
- **`model.eval()`** cambia el modo de capas como dropout; **`torch.no_grad()`** deja de retener memoria para `backward()`. Son independientes y para evaluar se usan los dos.

## Ejercicios

1. Calcula a mano cuántos parámetros tiene una red 784 → 256 → 64 → 10 y compruébalo con `sum(p.numel() for p in modelo.parameters())`. ¿Qué fracción está en la primera capa?
2. Escribe la clase `Red` sin `nn.Flatten`, aplanando dentro de `forward` con `x.flatten(1)`. ¿Cambia el número de parámetros? ¿Por qué?
3. Sin ejecutar: si el lote tuviera 32 imágenes, ¿qué números aparecerían en el error de la sección 6? Compruébalo.
4. Crea la capa de salida dentro de `forward` (`return nn.Linear(128, 10)(x)`) en vez de en `__init__`. ¿Cuántos parámetros registra el modelo? ¿Qué pasa si llamas dos veces al modelo con la misma imagen?
5. Agrega un `nn.Dropout(0.5)` a la clase `Red`, después de la ReLU. ¿Cambia el número de parámetros? ¿Por qué?
6. Usa `memoria_retenida` con lotes de 10, 100 y 1.000 imágenes. ¿Cómo crece la memoria retenida con el tamaño del lote?